In [ ]:
# Install the required packages for Vespa Python client and CLI
#!pip3 install pyvespa vespacli

In [ ]:


import random
import pickle
from vespa.deployment import VespaCloud
import json
import unicodedata
from dataclasses import dataclass
from typing import Callable, Optional, Iterable, Dict
from vespa.application import Vespa
from time import time
from tqdm.auto import tqdm
import nest_asyncio
from vespa.evaluation import VespaEvaluator
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
)

In [ ]:
# nosso tenant name
tenant_name = "segundoprojetoemcd"
# nome da aplicação
application = "myapp"

In [ ]:
# Aqui podemos alterar o esquema caso necessário, mas por enquanto vamos usar o esquema padrão do tutorial.
package = ApplicationPackage(
    name=application,
    schema=[
        Schema(
            name="doc",
            document=Document(
                fields=[
                    Field(name="id", type="string", indexing=["summary"]),
                    Field(
                        name="body",
                        type="string",
                        indexing=["index", "summary"],
                        index="enable-bm25",
                        bolding=True,
                    ),
                    Field(
                        name="embedding",
                        type="tensor(x[384])",
                        indexing=[
                            'input body',
                            "embed",
                            "index",
                            "attribute",
                        ],
                        ann=HNSW(distance_metric="angular"),
                        is_document_field=False,
                    ),
                ]
            ),
            fieldsets=[FieldSet(name="default", fields=["body"])],
            rank_profiles=[
                RankProfile(
                    name="bm25",
                    inputs=[("query(q)", "tensor(x[384])")],
                    functions=[
                        Function(name="bm25sum", expression="bm25(body)")
                    ],
                    first_phase="bm25sum",
                ),
                RankProfile(
                    name="semantic",
                    inputs=[("query(q)", "tensor(x[384])")],
                    first_phase="closeness(field, embedding)",
                ),
                RankProfile(
                    name="fusion",
                    inherits="bm25",
                    inputs=[("query(q)", "tensor(x[384])")],
                    first_phase="closeness(field, embedding)",
                    global_phase=GlobalPhaseRanking(
                        expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                        rerank_count=1000,
                    ),
                ),
                RankProfile(
                    name="weighted_hybrid",
                    inherits="bm25", # To get bm25sum function
                    inputs=[("query(q)", "tensor(x[384])")],
                    functions=[
                        Function(
                            name="normalized_bm25",
                            expression="atan(bm25sum / 12) / (3.14159 / 2)"
                        ),
                        Function(
                            name="semantic_score",
                            expression="closeness(field, embedding)"
                        )
                    ],
                    first_phase=" (0.7 * normalized_bm25) + (0.3 * semantic_score) "
                    )
            ],
        )
    ],
    components=[
        Component(
            id="e5",
            type="hugging-face-embedder",
            parameters=[
                Parameter(
                    "transformer-model",
                    {
                        "url": "https://github.com/vespa-engine/sample-apps/raw/master/examples/model-exporting/model/e5-small-v2-int8.onnx"
                    },
                ),
                Parameter(
                    "tokenizer-model",
                    {
                        "url": "https://raw.githubusercontent.com/vespa-engine/sample-apps/master/examples/model-exporting/model/tokenizer.json"
                    },
                ),
            ],
        )
    ],
)

In [ ]:
# Sempre que essa célula der erro execute o comando vespa auth login no terminal
# para autenticar novamente com o Vespa Cloud
vespa_cloud = VespaCloud(
    tenant=tenant_name,
    application=application,
    application_package=package,
)

In [ ]:
#essa função simplifica o processo de obtenção ou criação da aplicação no Vespa Cloud
def get_or_deploy_application(vespa_cloud):
    from urllib3.exceptions import HTTPError

    try:
        return vespa_cloud.get_application()
    except HTTPError as e:
        if "NOT_FOUND" in str(e):
            return vespa_cloud.deploy()
        raise

app = get_or_deploy_application(vespa_cloud)


In [ ]:
# test to verify the connection and deployment
# The expected output is:Found mtls endpoint for myapp_container

endpoint = vespa_cloud.get_mtls_endpoint()
endpoint

In [ ]:
# Splita as queries de treinamento e teste

random.seed(42)

def load_dataset(input_file):
    with open(input_file, 'rb') as f:
        return pickle.load(f)

data_set = "subset_msmarco_train_0/subset_msmarco_train_0.01_99.pkl"

data = load_dataset(data_set)
queries = data["queries"]
documents = data["docs"]
qrels = data["qrels"]

# Split the queries (queries is a dictionary of {query_id: query_object})
query_ids = list(queries.keys())  # List of query IDs

# Shuffle query IDs to ensure a random split
random.shuffle(query_ids)

# Split into 80% for training, 20% for validation
split_ratio = 0.8
train_query_ids = query_ids[:int(len(query_ids) * split_ratio)]
test_query_ids = query_ids[int(len(query_ids) * split_ratio):]

train_queries = {qid: queries[qid] for qid in train_query_ids}
test_queries = {qid: queries[qid] for qid in test_query_ids}

In [ ]:
# remove control characters from the text to avoid issues with Vespa Feed

def remove_control_characters(text: str) -> str:
    """Remove caracteres de controle e não imprimíveis do texto."""
    return ''.join(
        ch for ch in text
        if unicodedata.category(ch)[0] != 'C' or ch in '\n\t\r'
    )

In [ ]:
# Saves the data into a json format that Vespa can understand possibly we can remove this saving step 
# and create the json on memory just before the feed step

namespace = "default"
doctype = "doc"

vespa_docs = []

for doc_id, doc_obj in documents.items():
    vespa_doc = {
        "put": f"id:{namespace}:{doctype}::{doc_id}",
        "fields": {
            "id": str(doc_id),
            "body": remove_control_characters(doc_obj.text),
        }
    }
    vespa_docs.append(vespa_doc)

feed_file = "vespa_feed.json"

with open(feed_file, "w", encoding="utf-8") as f:
    json.dump(vespa_docs, f, ensure_ascii=False)

print(f"✅ {len(vespa_docs)} documentos salvos em {feed_file}")

In [ ]:
# Define feed parameters for the Vespa application
@dataclass
class FeedParams:
    name: str
    num_docs: int
    max_connections: int
    function_name: str
    max_workers: Optional[int] = None
    max_queue_size: Optional[int] = None


@dataclass
class FeedResult(FeedParams):
    feed_time: Optional[float] = None

In [ ]:
# This is necessary to avoid issues with asyncio and Jupyter notebooks
nest_asyncio.apply()

In [ ]:
# Asynchronous feed function that sends documents to Vespa
# params = FeedParams(
#     name="full_async_feed",
#     function_name="feed_async_iterable",
#     num_docs=0,  # não é usado aqui
#     max_connections=16,
#     max_workers=32,
#     max_queue_size=5000,
# )

# # Load the documents from the JSON file
# with open("vespa_feed.json", "r", encoding="utf-8") as f:
#     data_list = json.load(f)

# # Prepare the dataset for feeding into Vespa
# dataset = [
#     {"id": item["fields"]["id"], "fields": item["fields"]}
#     for item in tqdm(data_list, desc="🔄 Preparando documentos para envio")
# ]

# # Feed the dataset into Vespa asynchronously
# with tqdm(total=len(dataset), desc="📤 Enviando documentos para Vespa") as pbar:
#     def progress_callback(response, doc_id):
#         pbar.update(1)
#         if not response.is_successful():
#             print(f"❌ Erro ao enviar {doc_id}: {response.status_code} - {response.get_json()}")

#     start = time()
#     app.feed_async_iterable(
#         dataset,
#         schema="doc",
#         namespace="pyvespa-feed",
#         operation_type="feed",
#         max_queue_size=params.max_queue_size,
#         max_workers=params.max_workers,
#         max_connections=params.max_connections,
#         callback=progress_callback,
#     )
#     duration = time() - start

# print(f"✅ Feed finalizado em {duration:.2f} segundos") 


In [ ]:
# This cell performs the evaluation of the queries using the VespaEvaluator class

test_queries_dict = {
    q.query_id: q.text
    for q in test_queries.values()
}

relevant_docs = dict()
for qrel in qrels:
    relevant_docs[qrel.query_id] = relevant_docs.get(qrel.query_id, set())
    relevant_docs[qrel.query_id].add(qrel.doc_id)

def query_fn_for_ranking(ranking: str) -> callable:
    if ranking== "bm25":
        def query_fn(query_text: str, top_k: int) -> dict:
            return {
                "yql": f"select * from sources * where userQuery()",
                "query": query_text,
                "hits": top_k,
                "ranking": ranking,
            }
        return query_fn
    elif ranking == "semantic":
        def query_fn(query_text: str, top_k: int) -> dict:
            return {
                "yql": "select * from sources * where ({targetHits:10}nearestNeighbor(embedding,q))",
                "query": query_text,
                "hits": top_k,
                "ranking": ranking,
                "ranking.features.query(q)": f"embed({query_text})"
            }
        return query_fn
    elif ranking == "fusion":
        def query_fn(query_text: str, top_k: int) -> dict:
            return {
                # There is two options mentioned in the tutorial for fusion ranking.
                #yql="select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
                "yql":"select * from sources * where rank({targetHits:1000}nearestNeighbor(embedding,q), userQuery())",
                "query": query_text,
                "hits": top_k,
                "ranking": ranking,
                "ranking.features.query(q)": f"embed({query_text})"
            }
        return query_fn
    elif ranking == "bm_25_or_semantic":
        def query_fn(query_text: str, top_k: int) -> dict:
            return {
                "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
                "query": query_text,
                "hits": top_k,
                "ranking": "fusion",
                "ranking.features.query(q)": f"embed({query_text})"
            }
        return query_fn
    elif ranking == "weighted_hybrid":
        def query_fn(query_text: str, top_k: int) -> dict:
            return {
                "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
                "query": query_text,
                "hits": top_k,
                "ranking": ranking,
                "ranking.features.query(q)": f"embed({query_text})"
            }
        return query_fn


all_results = dict()
for ranking in ["bm25", "semantic", "fusion", "bm_25_or_semantic", "weighted_hybrid"]:
    query_fn = query_fn_for_ranking(ranking)
    evaluator = VespaEvaluator(
        queries=test_queries_dict,
        relevant_docs=relevant_docs,
        vespa_query_fn=query_fn,
        app=app,
        name=f"test-run-{ranking}",
        accuracy_at_k=[10],
        precision_recall_at_k=[10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[10],
        write_csv=True
    )
    
    results = evaluator.run()
    print(f"Results for {ranking}:")
    print("Primary metric:", evaluator.primary_metric)
    print("All results:", results)
    all_results[ranking] = results


In [ ]:
# transform all_results into a pandas DataFrame for better visualization
import pandas as pd
results_df = pd.DataFrame(all_results).T

results_df["mf1@10"] = results_df["precision@10"] * results_df["recall@10"] * 2 / (results_df["precision@10"] + results_df["recall@10"])

# Display the results DataFrame
results_df.round(3)

In [ ]:
# add mf1 score to the results DataFrame
results_df["mf1@10"] = results_df["precision@10"] * results_df["recall@10"] * 2 / (results_df["precision@10"] + results_df["recall@10"])

In [ ]:
results_df["mf1@10"].round(3)

In [ ]:
results_df['mrr@10']

In [ ]:
# Run BM25 over the training queries and collect score distributions

bm25_scores = []

def get_bm25_scores(query_text, top_k=100):
    query_body = {
        "yql": "select * from sources * where userQuery()",
        "query": query_text,
        "hits": top_k,
        "ranking": "bm25"
    }
    response = app.query(body=query_body)
    response = response.json
    if "root" in response and "children" in response["root"]:
        return [child["relevance"] for child in response["root"]["children"] if "relevance" in child]
    return []

# Get only the first 10% of the training queries
num_queries = int(len(train_queries) * 0.1)
first_10_percent = list(train_queries.items())[:num_queries]

for qid, query_obj in tqdm(first_10_percent, desc="Running BM25 on first 10% of training queries"):
    scores = get_bm25_scores(query_obj.text)
    bm25_scores.extend(scores)

print(f"Collected {len(bm25_scores)} BM25 scores from training queries.")

In [ ]:
# plot the distribution of BM25 scores
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.hist(bm25_scores, bins=50, color='blue', alpha=0.7)
plt.title("BM25 Score Distribution")
plt.xlabel("BM25 Score")
plt.ylabel("Frequency")
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# get the average BM25 score, and the median BM25 score
average_bm25_score = sum(bm25_scores) / len(bm25_scores)
median_bm25_score = sorted(bm25_scores)[len(bm25_scores) // 2]
print(f"Average BM25 Score: {average_bm25_score}")
print(f"Median BM25 Score: {median_bm25_score}")